In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import gc
from collections import defaultdict

years = np.arange(2008, 2016)
names = ['Attività e Passività', 'Costi e Ricavi', 'Entrate', 'Spese']

In [ ]:
columns = ['Denominazione', 'Dizione_Provincia', 'Dizione_Regione', 'Dizione_zona']

final = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

final['Data_Istituzione'] = pd.to_datetime(final['Data_Istituzione'], format='%Y-%m-%d')
final = final[final['Data_Istituzione'].dt.year < 2008]

final = final[final['Data_Cessazione'].isna()]

final = final[final['Codice_Tipologia_DLGS_118_2011'] == 'ELCOMU'][columns]

final = final.rename(columns={'Denominazione': 'Comune', 'Dizione_Provincia': 'Provincia', 'Dizione_Regione': 'Regione', 'Dizione_zona': 'Zona'})
final.index.names = ['BDAP']

final

In [ ]:
final.to_csv('comuni.csv')

In [ ]:
columns = ['Denominazione', 'Dizione_Provincia', 'Dizione_Regione', 'Dizione_zona']

df = pd.read_csv(
    'data//Anagrafe_comuni.csv',
    sep=';',
    encoding='latin_1',
    low_memory=False
).set_index('Id_Ente')

df['Data_Istituzione'] = pd.to_datetime(df['Data_Istituzione'], format='%Y-%m-%d', errors='coerce')
df['Data_Cessazione'] = pd.to_datetime(df['Data_Cessazione'], format='%Y-%m-%d', errors='coerce')

df = df[df['Codice_Tipologia_DLGS_118_2011'] == 'ELCOMU']

final_old = df[df['Data_Cessazione'].notna()][columns]
final_old = final_old.rename(columns={
    'Denominazione': 'Comune',
    'Dizione_Provincia': 'Provincia',
    'Dizione_Regione': 'Regione',
    'Dizione_zona': 'Zona'
})
final_old.index.names = ['BDAP']

final_new = df[df['Data_Istituzione'].dt.year >= 2008][columns]
final_new = final_new.rename(columns={
    'Denominazione': 'Comune',
    'Dizione_Provincia': 'Provincia',
    'Dizione_Regione': 'Regione',
    'Dizione_zona': 'Zona'
})
final_new.index.names = ['BDAP']

In [ ]:
final_old.to_csv('comuni_dismessi.csv')
final_new.to_csv('comuni_nuovi.csv')

# Zona

In [ ]:
zona = pd.DataFrame(index = final.index)
for elem in final['Zona'].unique():
	zona[elem] = 0
for i, row in final.iterrows():
	zona.loc[i, row['Zona']] = 1
	
zona.to_csv('zona.csv')

In [ ]:
zona = pd.read_csv('zona.csv', index_col='BDAP')


# Popolazione

In [ ]:
import os

def find_files(folder_path, search_string):
    matching_files = []

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if search_string in file:
                matching_files.append(os.path.join(root, file))

    return matching_files

def mean_age(data):
	 return np.mean(np.multiply(np.arange(0,101), data).sum())

In [ ]:
comuni = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

In [ ]:
popolazione = pd.DataFrame(index = final.index, columns = [f'pop_{year}' for year in years])

folder_path = 'popolazione//'

for idx in final.index:
	codice = str(comuni.loc[idx, 'Codice_ISTAT_Comune']).zfill(6)
	if codice == '099031':
		codice = '041060'
	if codice == '099030':
		codice = '041033'
	names = find_files(folder_path, codice)
	temp = pd.read_csv(names[0], sep = ';', encoding = 'latin_1', low_memory = False, skiprows=106, nrows=1).values[0][7:15]
	popolazione.loc[idx] = temp
	
popolazione = popolazione.apply(pd.to_numeric, errors='coerce')

In [ ]:
popolazione.to_csv('pop.csv')

# Balance sheet items

## Periodo 1

In [ ]:
pop = pd.read_csv('data/popolazione.csv', index_col='BDAP')

In [ ]:
name = 'Entrate'

filename = f'bilanci/{name}.csv'

bigdata = pd.read_csv(filename, sep = ';', encoding = 'latin_1', low_memory = False)
bigdata['Riscossione in CC'] = bigdata['Riscossione in CC'].str.replace(',', '.').astype(float)
bigdata['Riscossione in CR'] = bigdata['Riscossione in CR'].str.replace(',', '.').astype(float)
bigdata['Accertamento in CC'] = bigdata['Accertamento in CC'].str.replace(',', '.').astype(float)

In [ ]:
years = np.arange(2009, 2016)
target = ['Accertamento in CC', 'Riscossione in CC', 'Riscossione in CR']
for year in years:
    temp = bigdata[bigdata['Esercizio Finanziario'] == year]
    temp_df = temp.pivot_table(index='Codice BDAP Ente', columns = 'Descrizione Titolo', values = target, aggfunc='sum', fill_value=0)
    temp_df = temp_df.reindex(final.index, fill_value=0)
    unwanted = "TOTALE A PAREGGIO ( entrate tributarie + quota imu)"
    temp_df = temp_df.drop(columns=unwanted, level=1, errors='ignore')
    temp_df.columns = [f"{val} {titolo}" for val, titolo in temp_df.columns]
    temp_df = temp_df/pop[f'{year}'].values[:, np.newaxis]
    temp_df.to_csv(f'Entrate {year}.csv')
    print(f'{name} {year}\tShape: {temp_df.shape}\tNaN: {temp_df.isnull().sum().sum()} ')

del bigdata
del temp
gc.collect()

In [ ]:
name = 'Spese'

filename = f'bilanci/{name}.csv'

bigdata = pd.read_csv(filename, sep = ';', encoding = 'latin_1', low_memory = False)
bigdata['Impegno totale'] = bigdata['Impegno totale'].str.replace(',', '.').astype(float)
bigdata['Pagamenti in CR'] = bigdata['Pagamenti in CR'].str.replace(',', '.').astype(float)
bigdata['Pagamenti in CC'] = bigdata['Pagamenti in CC'].str.replace(',', '.').astype(float)

In [ ]:
years = np.arange(2009, 2016)
target = ['Impegno totale', 'Pagamenti in CC', 'Pagamenti in CR']
for year in years:
    temp = bigdata[bigdata['Esercizio Finanziario'] == year]
    temp_df = temp.pivot_table(index='Codice BDAP Ente', columns = 'Descrizione Titolo Spese', values = target, aggfunc='sum', fill_value=0)
    temp_df = temp_df.reindex(final.index, fill_value=0)
    unwanted = 'TOTALE A PAREGGIO ( spese correnti + spesa per alimentazione fondo)'
    temp_df = temp_df.drop(columns=unwanted, level=1, errors='ignore')
    temp_df.columns = [f"{val} {titolo}" for val, titolo in temp_df.columns]
    temp_df = temp_df/pop[f'{year}'].values[:, np.newaxis]
    temp_df.to_csv(f'Spese {year}.csv')
    print(f'{name} {year}\tShape: {temp_df.shape}\tNaN: {temp_df.isnull().sum().sum()} ')

del bigdata
del temp
gc.collect()

## Periodo 2

In [ ]:
years = np.arange(2016, 2024)
target = ['Impegni', 'Pagamenti in CC', 'Pagamenti in CR']
for year in years:
    filename = f'bilanci/{year}_Rendiconto - Schemi di bilancio/{year}_Rendiconto SDB Spese Riepilogo Titoli.csv'
    bigdata = pd.read_csv(filename, sep = ';', encoding = 'latin_1', low_memory = False)
    bigdata = bigdata.rename(columns={'Pagamenti in C/Competenza': 'Pagamenti in CC', 'Pagamenti in C/Residui': 'Pagamenti in CR'})
    temp = bigdata[bigdata['Esercizio Finanziario'] == year]
    temp_df = temp.pivot_table(index='Codice BDAP', columns = 'Descrizione Titolo', values = target, aggfunc='sum', fill_value=0)
    temp_df = temp_df.reindex(final.index, fill_value=0)
    unwanted = "Altre voci di squadratura"
    temp_df = temp_df.drop(columns=unwanted, level=1, errors='ignore')
    temp_df.columns = [f"{val} {titolo}" for val, titolo in temp_df.columns]
    temp_df = temp_df/pop[f'{year}'].values[:, np.newaxis]
    temp_df.to_csv(f'Spese {year}.csv')
    print(f'Spese {year}\tShape: {temp_df.shape}\tNaN: {temp_df.isnull().sum().sum()} ')


del bigdata
del temp
gc.collect()

In [ ]:
years = np.arange(2016, 2024)
target = ['Accertamenti', 'Riscossioni in CC', 'Riscossioni in CR']
for year in years:
    filename = f'bilanci/{year}_Rendiconto - Schemi di bilancio/{year}_Rendiconto SDB Entrate Riepilogo Titoli.csv'
    bigdata = pd.read_csv(filename, sep = ';', encoding = 'latin_1', low_memory = False)
    bigdata = bigdata.rename(columns={'Riscossioni in C/Competenza': 'Riscossioni in CC', 'Riscossioni in C/Residui': 'Riscossioni in CR'})
    temp = bigdata[bigdata['Esercizio Finanziario'] == year]
    temp_df = temp.pivot_table(index='Codice BDAP', columns = 'Descrizione Titolo', values = target, aggfunc='sum', fill_value=0)
    temp_df = temp_df.reindex(final.index, fill_value=0)
    unwanted = "Altre voci di squadratura"
    temp_df = temp_df.drop(columns=unwanted, level=1, errors='ignore')
    temp_df.columns = [f"{val} {titolo}" for val, titolo in temp_df.columns]
    temp_df = temp_df/pop[f'{year}'].values[:, np.newaxis]
    temp_df.to_csv(f'Entrate {year}.csv')
    print(f'Entrate {year}\tShape: {temp_df.shape}\tNaN: {temp_df.isnull().sum().sum()} ')

del bigdata
del temp
gc.collect()

# Indicators

In [ ]:
df = pd.read_excel('data/Indicatori_CdC.xlsx')

indicator_dfs = {}
r_cols = [f'R{i}' for i in range(1, 11)]

temp_dfs = {}
for year in df['eserciziofinanziario'].unique():
    filtered_df = df[df['eserciziofinanziario'] == year].copy()
    indicator_df = filtered_df.set_index('codicebdap')[r_cols].fillna(0)
    indicator_df.index.name = 'BDAP'
    temp_dfs[year] = indicator_df

common_index = set(final.index)
for indicator_df in temp_dfs.values():
    common_index &= set(indicator_df.index)
common_index = sorted(common_index)

for year, indicator_df in temp_dfs.items():
    indicator_df = indicator_df.loc[common_index]
    indicator_dfs[year] = indicator_df
    indicator_df.to_csv(f'Indicatori {year}.csv')
    print(f'Saved: Indicatori {year}.csv (Shape: {indicator_df.shape})')

# Anticipazioni

In [ ]:
pop = pd.read_csv('data//popolazione.csv', index_col= 'BDAP')
df = pd.read_excel('data//Indicatori_anticipazioni_tesoreria.xlsx')
anagrafe = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

indicator_dfs = {}
cols = [
    'giorni_tiraggio', 'tot_anticip_rich', 'num_anticipazioni',
    'durata_media_anticip', 'quota_utilizzo', 'anticip_media', 'max_cumulata'
]

temp_dfs = {}

for year in df['anno'].unique():
    if year in [2024, 2025, 2008]:
        continue

    filtered_df = df[df['anno'] == year].copy()

    rows = []   # collect rows here to avoid loc-assignment warning
    index_list = []

    for _, row in filtered_df.iterrows():
        t = anagrafe.loc[anagrafe['Codice_Ente_SIOPE'] == row['COD_ENTE']]
        if not t.empty:
            rows.append(row[cols].astype(float))
            index_list.append(t.index[0])

    # build the dataframe with correct dtypes
    temp_df = pd.DataFrame(rows, index=index_list)

    # compute A1..A5
    temp_df['A1'] = temp_df['durata_media_anticip'].fillna(0).astype(float)
    temp_df['A2'] = (temp_df['anticip_media'] / pop[f'{year}']).astype(float).fillna(0)
    temp_df['A3'] = (temp_df['max_cumulata'] / pop[f'{year}']).astype(float).fillna(0)
    temp_df['A4'] = temp_df['quota_utilizzo'].fillna(0).astype(float)
    temp_df['A5'] = (temp_df['tot_anticip_rich'] / pop[f'{year}']).astype(float).fillna(0)

    temp_df = temp_df[['A1', 'A2', 'A3', 'A4', 'A5']]

    # reindex at the end
    temp_df = temp_df.reindex(final.index, fill_value=0)

    temp_dfs[year] = temp_df
    temp_df.to_csv(f'Anticipazioni {year}.csv')

    print(f'Saved: Anticipazioni {year}.csv (Shape: {temp_df.shape})')

In [ ]:
pop = pd.read_csv('data//popolazione.csv', index_col= 'BDAP')
df = pd.read_excel('data//Indicatori_anticipazioni_tesoreria.xlsx')
anagrafe = pd.read_csv('data//Anagrafe_comuni.csv', sep = ';', encoding = 'latin_1', low_memory = False).set_index('Id_Ente')

indicator_dfs = {}
cols = ['giorni_tiraggio', 'tot_anticip_rich', 'num_anticipazioni', 'durata_media_anticip', 'quota_utilizzo', 'anticip_media', 'max_cumulata'] 

temp_dfs = {}
for year in df['anno'].unique():
    if year == 2024 or year == 2025 or year == 2008:
        continue
    temp_df = pd.DataFrame(columns = cols)
    filtered_df = df[df['anno'] == year].copy()

    for i, row in filtered_df.iterrows():
        t = anagrafe.loc[anagrafe['Codice_Ente_SIOPE'] == row['COD_ENTE']]
        temp_df.loc[t.index[0]] = row[cols]
    temp_df['A1'] = (temp_df['durata_media_anticip']).fillna(0)
    temp_df['A2'] = (temp_df['anticip_media'] / pop[f'{year}']).fillna(0)
    temp_df['A3'] = (temp_df['max_cumulata'] / pop[f'{year}']).fillna(0)
    temp_df['A4'] = (temp_df['quota_utilizzo']).fillna(0)
    temp_df['A5'] = (temp_df['tot_anticip_rich'] / pop[f'{year}']).fillna(0)
    temp_df = temp_df[['A1', 'A2', 'A3', 'A4', 'A5']]

    temp_df = temp_df.reindex(final.index, fill_value=0)

    temp_dfs[year] = temp_df
    temp_df.to_csv(f'Anticipazioni {year}.csv')
    print(f'Saved: Anticipazioni {year}.csv (Shape: {temp_df.shape})')
        